# 双路召回 + RRF 融合 + 重排序（教学笔记本）

本笔记把前两个笔记本（TF-IDF / BM25）串成一条完整的**生产级检索流水线**：

1. **双路召回**：稀疏路用 BM25（关键词精确匹配）+ 稠密路用向量检索（语义召回）；
2. **RRF 融合**：把两路排名合并成一份候选池；
3. **重排序**：先用加权打分做教学占位（第 4 节），再用真实 Cross-Encoder 对比（第 5 节）。

**依赖**：`rank_bm25` + `sentence-transformers`（`pip install rank_bm25 sentence-transformers`；首次运行会下载 `all-MiniLM-L6-v2` 和 `cross-encoder/ms-marco-MiniLM-L-6-v2`（各约 90MB））；稠密路使用真实语义向量（不再用 TF-IDF 占位）；
两路召回都会过滤 0 分文档，避免无关结果凭 RRF 排名分混入候选池。

## 0. 准备语料

一个多主题小语料，方便看出两路召回的差异（有的靠关键词、有的靠语义）。

In [1]:
import re

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

corpus = [
    "How to train a neural network with backpropagation",
    "BM25 is a probabilistic ranking function for search engines",
    "Dogs are loyal pets that need daily exercise and care",
    "Cats are independent animals that enjoy quiet environments",
    "Gradient descent optimizes model weights by minimizing loss",
    "Search engines use inverted indexes to retrieve relevant documents",
    "Puppies require training patience and consistent positive reinforcement",
    "Embeddings map text into vectors that capture semantic meaning",
]
print(f"语料共 {len(corpus)} 篇")

语料共 8 篇


## 1. 第 1 路：稀疏召回（BM25）

关键词精确匹配，对专有名词、数字、缩写很准，但无法处理同义/换说法。
本笔记直接使用 `rank_bm25` 的 `BM25Okapi`（`k1=1.5`、`b=0.75`），不再手写打分公式。
召回时只保留得分 > 0 的文档（0 分表示与查询没有任何词面重合）。

> **说明（停用词）**：本笔记的 `tokenize` 没有去停用词，`a` / `to` / `with` 这类高频词
> 也会贡献非零 BM25 分（示例中单次命中约 0.94 分），因此用词不相关的文档仍可能进入 Top-K。
> 这里刻意保留以展示 BM25 的原始行为；生产环境应加停用词表或设置最小分阈值。

In [2]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [tokenize(d) for d in corpus]
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

def bm25_scores(query):
    """返回 {doc_id: BM25 分数}，查询与建库使用同一套分词。"""
    return {i: float(s) for i, s in enumerate(bm25.get_scores(tokenize(query)))}

def sparse_retrieve(query, top_k=5):
    # 0 分过滤：只保留至少命中一个查询词的文档
    hits = [(i, s) for i, s in bm25_scores(query).items() if s > 0]
    hits.sort(key=lambda x: x[1], reverse=True)
    return hits[:top_k]


## 2. 第 2 路：稠密召回（向量检索）

把文本编码成向量，用余弦相似度做最近邻检索，能召回**语义相关**但用词不同的文档。

本笔记用 `sentence-transformers` 的 **all-MiniLM-L6-v2** 编码文档与查询：
向量做 L2 归一化后，内积直接等于余弦相似度；同样过滤掉 0 分（及负分）文档。


In [3]:
# 真实语义向量：sentence-transformers 编码 + 余弦相似度检索
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
# normalize_embeddings=True：向量已 L2 归一化，点积即余弦相似度
doc_vecs = model.encode(corpus, normalize_embeddings=True).astype("float32")
print(f"文档向量矩阵: {doc_vecs.shape}")

def encode_query(query):
    """查询与文档用同一模型、同一归一化方式编码，保证在同一向量空间。"""
    return model.encode([query], normalize_embeddings=True)[0].astype("float32")

def dense_retrieve(query, top_k=5):
    qvec = encode_query(query)
    scores = doc_vecs @ qvec                      # 归一化后点积 = 余弦相似度
    scored = [(i, float(s)) for i, s in enumerate(scores) if s > 0]  # 0 分过滤
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


W1019 23:04:56.584000 78832 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

文档向量矩阵: (8, 384)


### 说明：embedding 模型与首次下载

- 模型在首次运行时自动下载，之后走本地缓存。
- 国内网络下载慢时，可在导入 `sentence_transformers` 前设置
  `os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"` 走镜像。
- 想换更强的语义模型，只需替换模型名（如 `BAAI/bge-small-zh-v1.5`），其余检索代码不变。


## 3. 合并候选 · RRF 融合

两路分数量纲不同，直接加权容易失衡。常用 **Reciprocal Rank Fusion** 只按「排名」融合：

$$RRF(d)=\sum_{r\in\text{各路}} \frac{1}{k+\,rank_r(d)}$$

`k` 一般取 60，作用是压平排名差距、不依赖各路原始分数尺度。融合后取 Top-M 作为重排候选池。
RRF 只给「出现在列表里」的文档加分，因此召回阶段的 0 分过滤能让候选池只保留真实命中的文档。

In [5]:
def rrf(ranked_lists, k=60):
    scores = {}
    for ranked in ranked_lists:
        for rank, (doc_id, _) in enumerate(ranked):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

query_demo = "how to train a model with gradient descent"
sp = sparse_retrieve(query_demo)
de = dense_retrieve(query_demo)
fused = rrf([sp, de])

print(f'查询: "{query_demo}"\n')
print("稀疏召回（0 分已过滤）:")
for i, s in sp:
    print(f"  doc{i} {s:.3f}  {corpus[i]}")
print("\n稠密召回（0 分已过滤）:")
for i, s in de:
    print(f"  doc{i} {s:.3f}  {corpus[i]}")
print("\nRRF 融合候选（按融合分降序）:")
for i, s in fused:
    print(f"  doc{i} rrf={s:.4f}  {corpus[i]}")

查询: "how to train a model with gradient descent"

稀疏召回（0 分已过滤）:
  doc0 6.967  How to train a neural network with backpropagation
  doc4 4.991  Gradient descent optimizes model weights by minimizing loss
  doc1 0.937  BM25 is a probabilistic ranking function for search engines
  doc5 0.937  Search engines use inverted indexes to retrieve relevant documents

稠密召回（0 分已过滤）:
  doc4 0.695  Gradient descent optimizes model weights by minimizing loss
  doc0 0.613  How to train a neural network with backpropagation
  doc7 0.227  Embeddings map text into vectors that capture semantic meaning
  doc6 0.200  Puppies require training patience and consistent positive reinforcement
  doc1 0.138  BM25 is a probabilistic ranking function for search engines

RRF 融合候选（按融合分降序）:
  doc0 rrf=0.0325  How to train a neural network with backpropagation
  doc4 rrf=0.0325  Gradient descent optimizes model weights by minimizing loss
  doc1 rrf=0.0313  BM25 is a probabilistic ranking function for search engines
  do

## 4. 重排序（Re-ranking）

融合只用了「排名」，丢掉了各路的**细粒度交互分数**。重排序阶段用更强的模型对候选池精排：

- **生产做法**：Cross-Encoder 把 `(query, doc)` **拼在一起**喂进 Transformer，直接输出相关性分数；
  或让 LLM 做 pairwise / listwise 排序。它们比 bi-encoder 准得多，但逐对计算很慢，所以**只用于少量候选**。
- **本笔记占位**：用归一化后的 `α·BM25分 + (1−α)·稠密余弦` 对融合候选重新打分，
  演示「用更丰富信号精排」这一思想；真实 Cross-Encoder 只是把这里的手工权重换成了神经网络学到的交互。

第 5 节会保留这套公式方案，并额外给出真实 Cross-Encoder 的对比案例。

In [6]:
def rerank(query, candidate_ids, top_n=3, alpha=0.5):
    sparse_all = bm25_scores(query)
    positive = [s for s in sparse_all.values() if s > 0]
    max_s = max(positive) if positive else 1.0
    qvec = encode_query(query)
    re = []
    for i in candidate_ids:
        ns = sparse_all.get(i, 0.0) / max_s          # BM25 分归一化到 0..1
        nd = float(doc_vecs[i] @ qvec)               # 余弦相似度（通常为正，也可能为负）
        re.append((i, alpha * ns + (1 - alpha) * nd))
    re.sort(key=lambda x: x[1], reverse=True)
    return re[:top_n]


## 5. 真实 Cross-Encoder 重排序（对比案例）

第 4 节的加权公式只是教学占位：BM25 分和稠密余弦各自独立计算，再手工加权。
生产中的重排序通常用 **Cross-Encoder**：把 `(query, doc)` 拼成一对喂进 Transformer，
让模型直接建模两者的交互并输出相关性分数。它逐对计算、比 bi-encoder 慢，所以只用于少量候选。

这里用轻量的 **cross-encoder/ms-marco-MiniLM-L-6-v2**：6 层 MiniLM、约 22.7M 参数、
权重约 90MB，CPU 上即可运行（首次运行会自动下载）。若想更省资源，可以换成
`cross-encoder/ms-marco-TinyBERT-L-2-v2`（约 4M 参数，精度略降）。

注意：Cross-Encoder 输出的是相关性 **logit**（可正可负），只在同一模型内比较大小，
不等于概率；需要 0..1 分数时可以再套一层 sigmoid。


In [7]:
from sentence_transformers import CrossEncoder

# 轻量 Cross-Encoder：6 层 MiniLM，CPU 可跑
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def cross_encoder_rerank(query, candidate_ids, top_n=3):
    pairs = [(query, corpus[i]) for i in candidate_ids]
    scores = cross_encoder.predict(pairs)           # 相关性 logit，越大越相关
    ranked = sorted(zip(candidate_ids, scores), key=lambda x: x[1], reverse=True)
    return [(i, float(s)) for i, s in ranked][:top_n]

candidate_ids = [i for i, _ in fused[:6]]           # 与端到端流程相同的候选池
formula_top = rerank(query_demo, candidate_ids)     # 第 4 节的加权公式
ce_top = cross_encoder_rerank(query_demo, candidate_ids)

print(f'候选池（RRF Top-{len(candidate_ids)}）:')
for i in candidate_ids:
    print(f"  doc{i}  {corpus[i]}")
print("\n加权公式重排 Top-3:")
for i, s in formula_top:
    print(f"  doc{i} score={s:.4f}  {corpus[i]}")
print("\nCross-Encoder 重排 Top-3:")
for i, s in ce_top:
    print(f"  doc{i} logit={s:+.4f}  {corpus[i]}")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

候选池（RRF Top-6）:
  doc0  How to train a neural network with backpropagation
  doc4  Gradient descent optimizes model weights by minimizing loss
  doc1  BM25 is a probabilistic ranking function for search engines
  doc7  Embeddings map text into vectors that capture semantic meaning
  doc5  Search engines use inverted indexes to retrieve relevant documents
  doc6  Puppies require training patience and consistent positive reinforcement

加权公式重排 Top-3:
  doc0 score=0.8063  How to train a neural network with backpropagation
  doc4 score=0.7060  Gradient descent optimizes model weights by minimizing loss
  doc1 score=0.1361  BM25 is a probabilistic ranking function for search engines

Cross-Encoder 重排 Top-3:
  doc4 logit=+3.9710  Gradient descent optimizes model weights by minimizing loss
  doc0 logit=-9.9019  How to train a neural network with backpropagation
  doc6 logit=-11.3721  Puppies require training patience and consistent positive reinforcement


> **结果解读**：同一个候选池下，加权公式把仅靠停用词 `a` 命中 BM25 的 `doc1` 排进了 Top-3；
> Cross-Encoder 把它挤出了 Top-3，并把语义相关的 `doc6`（Puppies require training...）提了上来。
> 注意 `doc0` 的 logit 为负仍排第 2：logit 的绝对值不是概率，只在同一模型内比较相对大小。
> 这正是「Cross-Encoder 更准但更慢，只用于少量候选」的直观体现。


## 6. 端到端串联 & 小结

把四步包成一个 `hybrid_search`，跑一个查询看全链路：召回 → 融合 → 重排。

In [8]:
def hybrid_search(query, recall_k=5, top_m=6, top_n=3):
    sp = sparse_retrieve(query, recall_k)
    de = dense_retrieve(query, recall_k)
    fused = rrf([sp, de])
    cand_ids = [i for i, _ in fused[:top_m]]
    final = rerank(query, cand_ids, top_n)
    return sp, de, fused, final

query = "how to train a model with gradient descent"
sp, de, fused, final = hybrid_search(query)

print(f'=== 端到端查询: "{query}" ===\n')
print("① 稀疏召回:")
for i, s in sp:
    print(f"  doc{i} {s:.3f}  {corpus[i]}")
print("\n② 稠密召回:")
for i, s in de:
    print(f"  doc{i} {s:.3f}  {corpus[i]}")
print("\n③ RRF 融合候选:")
for i, s in fused:
    print(f"  doc{i} rrf={s:.4f}  {corpus[i]}")
print("\n④ 重排序 Top-3:")
for i, s in final:
    print(f"  doc{i} score={s:.4f}  {corpus[i]}")

=== 端到端查询: "how to train a model with gradient descent" ===

① 稀疏召回:
  doc0 6.967  How to train a neural network with backpropagation
  doc4 4.991  Gradient descent optimizes model weights by minimizing loss
  doc1 0.937  BM25 is a probabilistic ranking function for search engines
  doc5 0.937  Search engines use inverted indexes to retrieve relevant documents

② 稠密召回:
  doc4 0.695  Gradient descent optimizes model weights by minimizing loss
  doc0 0.613  How to train a neural network with backpropagation
  doc7 0.227  Embeddings map text into vectors that capture semantic meaning
  doc6 0.200  Puppies require training patience and consistent positive reinforcement
  doc1 0.138  BM25 is a probabilistic ranking function for search engines

③ RRF 融合候选:
  doc0 rrf=0.0325  How to train a neural network with backpropagation
  doc4 rrf=0.0325  Gradient descent optimizes model weights by minimizing loss
  doc1 rrf=0.0313  BM25 is a probabilistic ranking function for search engines
  doc7 rrf=

## 小结

- **双路召回**用「精确 + 语义」互补，覆盖面远大于单路；
- **0 分过滤**保证 RRF 只在真实召回的文档之间做排名融合；
- **RRF** 用排名融合，避免两路分数尺度不一致的问题；
- **重排序**用更强但更贵的模型，只对少量候选精排——既保召回率又保精度，还控制算力；
- 第 2 路已使用真实 embedding（可替换为领域内更强的模型）；第 5 节已用轻量 Cross-Encoder 实测，生产中还可换 bge-reranker / Cohere Rerank 等更强的模型。